# Offline Batch Evaluation

Notebook nay chi dung de eval sau infer. No doc format batch dang co: `batch_root/run_XXXX.../<config>_seed<seed>/sample_*.png` va `final_rewards.json`.

Output gom 3 cap:
- `eval_per_image.csv`: score tung anh.
- `eval_run_summary.csv`: statistics tung run voi `mean`, `std`, `min`, `max`.
- `eval_batch_summary_from_run_stats.csv`: statistics toan batch tinh tren cac statistics cua tung run.

In [ ]:
from pathlib import Path

# Tro den batch root hoac danh sach batch roots.
EVAL_ROOTS = [Path(r"../logs/sb_batch/batch_20260424_111205_6101")]

# Neu None, prompt duoc doc tu final_rewards.json cua tung run.
# Set string neu folder anh khong co final_rewards.json.
PROMPT_OVERRIDE = None

REWARD_NAMES = ["clip", "pick", "image_reward"]
DEVICE = "cuda"
BATCH_SIZE = 4
IMAGE_GLOB = "sample_*.png"

PER_IMAGE_CSV = Path("eval_per_image.csv")
RUN_SUMMARY_CSV = Path("eval_run_summary.csv")
BATCH_SUMMARY_CSV = Path("eval_batch_summary_from_run_stats.csv")

In [ ]:
import csv
import json
import sys
from collections import defaultdict

import numpy as np
import torch
from PIL import Image

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from seg.scorers.ImageReward_scorer import ImageRewardScorer
from seg.scorers.PickScore_scorer import PickScoreScorer
from seg.scorers.clip_scorer import CLIPScorer

In [ ]:
def build_reward_scorer(name, dtype, device):
    normalized = name.lower()
    if normalized in {"clip", "clip_score"}:
        return CLIPScorer(dtype=dtype, device=device)
    if normalized in {"pick", "pick_score"}:
        return PickScoreScorer(dtype=dtype, device=device)
    if normalized in {"image_reward", "imagereward", "image_reward_score"}:
        return ImageRewardScorer(dtype=dtype, device=device)
    raise ValueError(f"Unsupported reward scorer: {name}")


def resolve_path(path):
    path = Path(path)
    return path.resolve() if path.is_absolute() else (REPO_ROOT / path).resolve()


def discover_eval_runs(roots, image_glob):
    """Return folders that represent one prompt/run: contain final_rewards.json and images."""
    eval_dirs = []
    for root in roots:
        root = resolve_path(root)
        if (root / "final_rewards.json").exists() and list(root.glob(image_glob)):
            eval_dirs.append(root)
            continue
        for metadata_path in root.rglob("final_rewards.json"):
            run_dir = metadata_path.parent
            if list(run_dir.glob(image_glob)):
                eval_dirs.append(run_dir)
    return sorted(set(eval_dirs))


def load_metadata(run_dir):
    metadata_path = run_dir / "final_rewards.json"
    if not metadata_path.exists():
        return {}
    with metadata_path.open("r", encoding="utf-8") as f:
        return json.load(f)


def prompt_for_run(run_dir, metadata):
    if PROMPT_OVERRIDE is not None:
        return PROMPT_OVERRIDE
    prompt = metadata.get("prompt")
    if prompt:
        return prompt
    raise ValueError(f"No prompt found for {run_dir}. Set PROMPT_OVERRIDE or keep final_rewards.json next to images.")


def batch_run_name(run_dir):
    # For .../batch_root/run_0000_xxx/pick_seed42, keep run_0000_xxx as run id.
    if run_dir.parent.name.startswith("run_"):
        return run_dir.parent.name
    return run_dir.name


def load_image_tensor(path):
    image = Image.open(path).convert("RGB")
    array = np.asarray(image, dtype=np.float32) / 255.0
    return torch.from_numpy(array).permute(2, 0, 1)


def summarize(values):
    values = np.asarray(values, dtype=np.float64)
    if values.size == 0:
        return {"count": 0, "mean": np.nan, "std": np.nan, "min": np.nan, "max": np.nan}
    return {
        "count": int(values.size),
        "mean": float(values.mean()),
        "std": float(values.std(ddof=0)),
        "min": float(values.min()),
        "max": float(values.max()),
    }

In [ ]:
eval_dirs = discover_eval_runs(EVAL_ROOTS, IMAGE_GLOB)
print(f"Found {len(eval_dirs)} eval runs")
for run_dir in eval_dirs[:10]:
    print(run_dir)

In [ ]:
device = torch.device(DEVICE if DEVICE.startswith("cuda") and torch.cuda.is_available() else "cpu")
dtype = torch.float16 if device.type == "cuda" else torch.float32
print(f"Using device={device}, dtype={dtype}")

scorers = {name: build_reward_scorer(name, dtype=dtype, device=device) for name in REWARD_NAMES}

In [ ]:
per_image_rows = []
run_summary_rows = []

for eval_dir in eval_dirs:
    metadata = load_metadata(eval_dir)
    prompt = prompt_for_run(eval_dir, metadata)
    run_id = batch_run_name(eval_dir)
    config = metadata.get("config", eval_dir.name)
    image_files = metadata.get("image_files") or [path.name for path in sorted(eval_dir.glob(IMAGE_GLOB))]
    image_paths = [eval_dir / name for name in image_files if (eval_dir / name).exists()]
    scores_by_reward = defaultdict(list)

    for offset in range(0, len(image_paths), BATCH_SIZE):
        batch_paths = image_paths[offset : offset + BATCH_SIZE]
        images = torch.stack([load_image_tensor(path) for path in batch_paths]).to(device=device)
        prompts = [prompt] * len(batch_paths)

        with torch.no_grad():
            batch_scores_by_reward = {}
            for reward_name, scorer in scorers.items():
                scores = scorer(images, prompts).detach().float().cpu().tolist()
                batch_scores_by_reward[reward_name] = scores
                scores_by_reward[reward_name].extend(scores)

        for local_idx, image_path in enumerate(batch_paths):
            row = {
                "run_id": run_id,
                "eval_dir": str(eval_dir),
                "config": config,
                "image": image_path.name,
                "prompt": prompt,
            }
            for reward_name in REWARD_NAMES:
                row[reward_name] = batch_scores_by_reward[reward_name][local_idx]
            per_image_rows.append(row)

    for reward_name in REWARD_NAMES:
        stats = summarize(scores_by_reward[reward_name])
        run_summary_rows.append({
            "run_id": run_id,
            "eval_dir": str(eval_dir),
            "config": config,
            "prompt": prompt,
            "reward": reward_name,
            **stats,
        })

print(f"Scored {len(per_image_rows)} images across {len(eval_dirs)} runs")

In [ ]:
batch_summary_rows = []
run_stat_names = ["mean", "std", "min", "max"]

# Statistics toan batch duoc tinh tu statistics tung run, khong tinh truc tiep tren tung anh.
for reward_name in REWARD_NAMES:
    reward_rows = [row for row in run_summary_rows if row["reward"] == reward_name]
    for run_stat in run_stat_names:
        values = [row[run_stat] for row in reward_rows]
        stats = summarize(values)
        batch_summary_rows.append({
            "reward": reward_name,
            "run_stat": run_stat,
            "num_runs": stats.pop("count"),
            **stats,
        })

batch_summary_rows

In [ ]:
per_image_fields = ["run_id", "eval_dir", "config", "image", "prompt", *REWARD_NAMES]
with PER_IMAGE_CSV.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=per_image_fields)
    writer.writeheader()
    writer.writerows(per_image_rows)

run_summary_fields = ["run_id", "eval_dir", "config", "prompt", "reward", "count", "mean", "std", "min", "max"]
with RUN_SUMMARY_CSV.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=run_summary_fields)
    writer.writeheader()
    writer.writerows(run_summary_rows)

batch_summary_fields = ["reward", "run_stat", "num_runs", "mean", "std", "min", "max"]
with BATCH_SUMMARY_CSV.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=batch_summary_fields)
    writer.writeheader()
    writer.writerows(batch_summary_rows)

print(f"Saved per-image scores: {PER_IMAGE_CSV.resolve()}")
print(f"Saved run summary: {RUN_SUMMARY_CSV.resolve()}")
print(f"Saved batch summary from run stats: {BATCH_SUMMARY_CSV.resolve()}")

In [ ]:
try:
    import pandas as pd

    display(pd.DataFrame(run_summary_rows).head())
    display(pd.DataFrame(batch_summary_rows))
except ImportError:
    print("Run summary sample:")
    for row in run_summary_rows[:5]:
        print(row)
    print("Batch summary:")
    for row in batch_summary_rows:
        print(row)